Week 3 Mortgage Rate Enrichment

"""
This script:
1. Fetches the MORTGAGE30US series directly from FRED
2. Resamples it from weekly to monthly averages
3. Merges it onto both the combined sold and listing datasets using a
   year_month key
4. Validates the merge (confirms no null rate values)
5. Saves both enriched datasets as new CSVs
"""

In [1]:
import pandas as pd
 
# -----------------------------------------------------------
# Step 0 - Load the datasets from Week 2-3 Deliverable A
# -----------------------------------------------------------
sold = pd.read_csv('sold_residential_filtered.csv', low_memory=False)
listing = pd.read_csv('listing_residential_filtered.csv', low_memory=False)
 
print(f"Sold rows loaded:    {len(sold)}")
print(f"Listing rows loaded: {len(listing)}")

Sold rows loaded:    448091
Listing rows loaded: 614541


In [9]:
# -----------------------------------------------------------
# Step 1 - Fetch the mortgage rate data from FRED
# -----------------------------------------------------------
url = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=MORTGAGE30US"
mortgage = pd.read_csv(url, parse_dates=['observation_date']) #turn into actual date 
mortgage.columns = ['date', 'rate_30yr_fixed']
 
print(f"\nFetched {len(mortgage)} weekly mortgage rate observations from FRED")
print(f"Date range: {mortgage['date'].min()} to {mortgage['date'].max()}")


Fetched 2885 weekly mortgage rate observations from FRED
Date range: 1971-04-02 00:00:00 to 2026-07-09 00:00:00


In [10]:
# -----------------------------------------------------------
# Step 2 - Resample weekly rates to monthly averages
# -----------------------------------------------------------
mortgage['year_month'] = mortgage['date'].dt.to_period('M')
mortgage_monthly = (
    mortgage.groupby('year_month')['rate_30yr_fixed']
    .mean()
    .reset_index()
)
 
print(f"\nResampled to {len(mortgage_monthly)} monthly averages")


Resampled to 664 monthly averages


In [11]:
# -----------------------------------------------------------
# Step 3 - Create a matching year_month key on the MLS datasets
# -----------------------------------------------------------
# Sold dataset -- key off CloseDate
sold['year_month'] = pd.to_datetime(sold['CloseDate'], errors='coerce').dt.to_period('M')
 
# Listing dataset -- key off ListingContractDate
listing['year_month'] = pd.to_datetime(
    listing['ListingContractDate'], errors='coerce'
).dt.to_period('M')

In [12]:
# -----------------------------------------------------------
# Step 4 - Merge
# -----------------------------------------------------------
sold_with_rates = sold.merge(mortgage_monthly, on='year_month', how='left')
listing_with_rates = listing.merge(mortgage_monthly, on='year_month', how='left')

In [13]:
# -----------------------------------------------------------
# Step 5 - Validate the merge
# -----------------------------------------------------------
sold_nulls = sold_with_rates['rate_30yr_fixed'].isnull().sum()
listing_nulls = listing_with_rates['rate_30yr_fixed'].isnull().sum()
 
print(f"\n--- Merge Validation ---")
print(f"Unmatched (null rate) rows in sold:    {sold_nulls}")
print(f"Unmatched (null rate) rows in listing: {listing_nulls}")
 
if sold_nulls > 0:
    print("\nSold records with no matching rate (check these year_month values):")
    print(sold_with_rates[sold_with_rates['rate_30yr_fixed'].isnull()]['year_month'].value_counts())
 
if listing_nulls > 0:
    print("\nListing records with no matching rate (check these year_month values):")
    print(listing_with_rates[listing_with_rates['rate_30yr_fixed'].isnull()]['year_month'].value_counts())
    
# Preview
print("\n--- Preview: sold_with_rates ---")
print(sold_with_rates[['CloseDate', 'year_month', 'ClosePrice', 'rate_30yr_fixed']].head())
 
print("\n--- Preview: listing_with_rates ---")
print(listing_with_rates[['ListingContractDate', 'year_month', 'ListPrice', 'rate_30yr_fixed']].head())
 


--- Merge Validation ---
Unmatched (null rate) rows in sold:    0
Unmatched (null rate) rows in listing: 0

--- Preview: sold_with_rates ---
    CloseDate year_month  ClosePrice  rate_30yr_fixed
0  2024-01-26    2024-01    240000.0           6.6425
1  2024-01-05    2024-01    815000.0           6.6425
2  2024-01-05    2024-01    810000.0           6.6425
3  2024-01-30    2024-01    858000.0           6.6425
4  2024-01-29    2024-01   1890500.0           6.6425

--- Preview: listing_with_rates ---
  ListingContractDate year_month   ListPrice  rate_30yr_fixed
0          2024-01-01    2024-01   1340000.0           6.6425
1          2024-01-24    2024-01   2500000.0           6.6425
2          2024-01-12    2024-01   3150000.0           6.6425
3          2024-01-20    2024-01   3090000.0           6.6425
4          2024-01-12    2024-01  12725000.0           6.6425


In [14]:
# -----------------------------------------------------------
# Step 6 - Save enriched datasets
# -----------------------------------------------------------
sold_with_rates.to_csv('sold_with_rates.csv', index=False, encoding='utf-8')
listing_with_rates.to_csv('listing_with_rates.csv', index=False, encoding='utf-8')
 
print(f"\nSaved sold_with_rates.csv ({len(sold_with_rates)} rows)")
print(f"Saved listing_with_rates.csv ({len(listing_with_rates)} rows)")


Saved sold_with_rates.csv (448091 rows)
Saved listing_with_rates.csv (614541 rows)
